In [1]:
import sys
import os
from pathlib import Path
import yaml

# 1. Ensure Python understands the project root so it can import modules from src
project_root = Path(os.getcwd())
# If running inside the notebooks folder, move up one level
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))


In [2]:
from src.discovery.mssql_inspector import MSSQLInspector
from src.utils.yaml_hydrator import YamlHydrator
from src.generators.hive_generator import HiveGenerator
from src.core.type_resolver import TypeResolver
from main import run_discovery, generate_scripts

In [3]:
config_path = project_root / "configs" / "sources" / "uat.yaml"
pipline_folder_path = project_root / "samples" / "pipelines"
schema_folder_path = project_root / "samples" / "metadata" / "schemas"
query_folder_path = project_root / "samples" / "metadata" / "query"
template_root= project_root / "template"
output_root= project_root / "samples" / "generated" / "datalake"

In [4]:
pipeline_config = generate_scripts('raw/m21_cashmovement')


[Step 1] Hydrating Metadata...
Hydration Success! Pipeline: raw_m21_cashmovement
Target Table: raw.m21_cashmovement
Template Path Prefix: datalake_model_5b

[Step 2] Generating Hive Scripts...

--- GENERATION SUCCESS ---
DDL File: C:\Users\dungp\projects\hql_spark_bridge\samples\generated\datalake\ddl_raw_m21_cashmovement.sql
DML File: C:\Users\dungp\projects\hql_spark_bridge\samples\generated\datalake\dml_raw_m21_cashmovement.sql
Hydration Success! Pipeline: com_t_m21_cashmovement
Target Table: com.t_m21_cashmovement
Template Path Prefix: datalake_model_5b

[Step 2] Generating Hive Scripts...

--- GENERATION SUCCESS ---
DDL File: C:\Users\dungp\projects\hql_spark_bridge\samples\generated\datalake\ddl_com_t_m21_cashmovement.sql
DML File: C:\Users\dungp\projects\hql_spark_bridge\samples\generated\datalake\dml_com_t_m21_cashmovement.sql


In [5]:
pipeline_config[1]

PipelineConfig(pipeline_id='com_t_m21_cashmovement', layer='com_t', model_type='5b', sources=[SourceModel(alias='src_m21', source_type='DB', connection_id='m21', query_path='metadata/discovery_output/queries/m21_cashmovement.sql', schema_path='m21_cashmovement.json', priority=1)], target=TargetModel(schema_name='com', table_name='t_m21_cashmovement', transformation_model='5b', partition_keys=['year_month'], primary_keys=[], date_keys=['TransactionDate']), columns=[ColumnModel(name='Firm', data_type='INTEGER', nullable=False, expression=None, is_partition=False, is_primary_key=False), ColumnModel(name='TransactionNo', data_type='INTEGER', nullable=False, expression=None, is_partition=False, is_primary_key=False), ColumnModel(name='TransactionDate', data_type='TIMESTAMP', nullable=False, expression=None, is_partition=False, is_primary_key=False), ColumnModel(name='ValueDate', data_type='TIMESTAMP', nullable=False, expression=None, is_partition=False, is_primary_key=False), ColumnModel(na

In [ ]:
TypeResolver.get_datatype(pipeline_config.columns[9], dialect='hive')

In [7]:
import re


def _replace_field_placeholders(expression: str, alias: str) -> str:
    """
    Finds placeholders like {{ field_name }} in the expression and replaces them
    with alias.field_name.
    """
    def replacer(match):
        field_name = match.group(1)
        return f"{alias}.{field_name}"

    # Regex to find {{ field_name }}
    # It captures 'field_name' in group 1
    return re.sub(r"\{\{\s*(\w+)\s*\}\}", replacer, expression)

_replace_field_placeholders("DATE_FORMAT({{ TransactionDate }}, 'yyyyMM')", 'T0')

"DATE_FORMAT(T0.TransactionDate, 'yyyyMM')"